# 8. LangChain and LangGraph

OpenBTK does not depend on LangChain. An **optional adapter** makes its
components usable inside LangChain and LangGraph applications.

Needs `pip install "openbtk[langchain,langgraph]"`.

> Every note, patient, number and identifier in this notebook is **fictitious**. It runs offline, downloads no model, and uses no real patient data.

In [1]:
import os

# Keep OpenBTK's routine debug lines out of this notebook's output.
os.environ.setdefault("OPENBTK_LOG_LEVEL", "warning")

'warning'

## De-identification as a Runnable

`as_runnable` wraps a component as a LangChain `Runnable`, so it composes with `|`.

In [2]:
from langchain_core.runnables import RunnableLambda

from openbtk.deid import DeidEngine, DeidMode
from openbtk.integrations.langchain import as_runnable

engine = DeidEngine(mode=DeidMode.REDACT, recognizers=["rule"])
deid_text = as_runnable(engine) | RunnableLambda(lambda result: result.text)

deid_text.invoke({"text": "Call (555) 010-2345 today.", "patient_id": "hashed-id"})

'Call [REDACTED] today.'

## Chunks and Documents

`chunk_to_document` / `document_to_chunk` convert losslessly. The adapter does not
de-identify anything - convert chunks of an already de-identified record.

In [3]:
from openbtk.core.schemas import TextSpan
from openbtk.data.clinical_text.schemas import ClinicalTextChunk
from openbtk.integrations.langchain import chunk_to_document, document_to_chunk

chunk = ClinicalTextChunk(
    chunk_id="note-1:0",
    record_id="note-1",
    text="Plan: rest and fluids.",
    span=TextSpan(start=0, end=22, label="chunk", confidence=1.0),
    token_count=6,
)
document = chunk_to_document(chunk)
assert document_to_chunk(document) == chunk  # nothing is lost
document.metadata

{'chunk_id': 'note-1:0',
 'record_id': 'note-1',
 'token_count': 6,
 'span_start': 0,
 'span_end': 22}

## A vector store LangChain can search

`OpenBTKVectorStore` presents any OpenBTK vector store as a LangChain `VectorStore`
(and `OpenBTKEmbeddings` / `OpenBTKChatModel` do the same for providers). Here it
wraps the FAISS store, so LangChain's own retriever works over it.

In [4]:
import re
import zlib

import numpy as np

from openbtk.core.base import BaseEmbeddingProvider
from openbtk.integrations.langchain import OpenBTKEmbeddings, OpenBTKVectorStore
from openbtk.retrieval.faiss import FAISSVectorStore


class HashingEmbedding(BaseEmbeddingProvider):
    """A lexical stand-in for a real embedding model."""

    sends_data_offsite = False
    dimension = 128

    def embed(self, texts):
        out = np.zeros((len(texts), self.dimension), dtype=np.float32)
        for i, text in enumerate(texts):
            for word in re.findall(r"[a-z0-9]+", text.lower()):
                out[i, zlib.crc32(word.encode()) % self.dimension] += 1.0
        norms = np.linalg.norm(out, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return out / norms


store = OpenBTKVectorStore(
    FAISSVectorStore(dimension=128, metric="ip"),
    OpenBTKEmbeddings(HashingEmbedding()),
)
store.add_texts(
    ["Type 2 diabetes, continue metformin.", "Hypertension controlled on lisinopril."],
    ids=["a", "b"],
)
retriever = store.as_retriever(search_kwargs={"k": 1})
retriever.invoke("metformin for diabetes")[0].page_content

'Type 2 diabetes, continue metformin.'

## An OpenBTK component as a LangGraph node

A LangGraph node is a callable from graph state to a partial update, so
`as_langgraph_node` returns exactly that.

In [5]:
from typing import Any, TypedDict

from langgraph.graph import END, START, StateGraph

from openbtk.integrations.langchain import as_langgraph_node


class State(TypedDict, total=False):
    request: dict
    result: Any


graph = StateGraph(State)
graph.add_node(
    "deidentify",
    as_langgraph_node(engine, input_key="request", output_key="result"),
)
graph.add_edge(START, "deidentify")
graph.add_edge("deidentify", END)

final = graph.compile().invoke(
    {
        "request": {
            "text": "Seen 03/14/2024. Call (555) 010-2345.",
            "patient_id": "hashed-id",
        }
    }
)
final["result"].text

'Seen [REDACTED]. Call [REDACTED].'

## Limits

`OpenBTKChatModel` does not stream token by token (OpenBTK's provider `stream()`
takes a prompt, not a message list), and `OpenBTKVectorStore` passes through the
store's own scores rather than normalised relevance scores. See the
[guide](https://openbtk.org/openbtk-core/dev/langchain/).